In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import sage.all as sage
from sage.all import var, sin, cos, pi, ln, e

In [ ]:
step = 0.025  # precision

tmin, tmax = -3, 1.5  # t bounds
xmin, xmax = -3, 3  # x bounds
ymin, ymax = -3, 3  # y bounds

compute_boundary = None  # Stop computation when values exceed this (None = no limit)

specific_solution = [] # to add specific solutions
ic_center = (0.0, 0.0, 0.0) # specify center or set to None for auto
ic_const = 0.9  # size of IC box (by proportion of volume)

num_st = 2  # number of solution curves on t-axis
num_sx = 8  # number of solution curves on x-axis
num_sy = 8  # number of solution curves on y-axis

point_size = 0.6  # size of inition condition points and Poincaré points

show_vectors = False  # show vectorfield ontop of solution curves
show_poincare_section = True  # display Poincaré section plane in 3D plot

# Poincaré section: specify a plane equation a*t + b*x + c*y + d = 0
poincare_a, poincare_b, poincare_c, poincare_d = 1, 0, 0, -3
flip_y = False # y orientation
flip_x = True # x orientation
eps = step  # tolerance for plane intersection + poincare analysis


In [ ]:
# SYSTEM DEFINITION

# f_x: R x R x R -> R
# dx/dt = f_x(t, x, y)
def f_x(t, x, y):
    if abs(x) <= 0.01 or abs(t) <= 0:
        return 0
    return (x / t) - (t / x)


# g_y: R x R x R -> R
# dy/dt = g_y(t, x, y)
def g_y(t, x, y):
    if abs(y) <= 0.01 or abs(t) <= 0:
        return 0
    return (y / t) - (t / y)

In [ ]:
# padding
pad_t, pad_x, pad_y = ((tmax - tmin) * 0.5, (xmax - xmin) * 0.5, (ymax - ymin) * 0.5)

# initial conditions
if ic_center is None:
    ics = [
        [sage.SR(i * ic_const), sage.SR(j * ic_const), sage.SR(k * ic_const)]
        for i in np.linspace(tmin, tmax + pad_t, num_st)
        for j in np.linspace(xmin, xmax + pad_x, num_sx)
        for k in np.linspace(ymin, ymax + pad_y, num_sy)
    ]
else:
    t_mid = ic_const * (abs(tmax) + abs(tmin)) / 2.0
    x_mid = ic_const * (abs(xmax) + abs(xmin)) / 2.0
    y_mid = ic_const * (abs(ymax) + abs(ymin)) / 2.0
    ics = [
        [sage.SR(i), sage.SR(j), sage.SR(k)]
        for i in np.linspace(
            ic_center[0] - t_mid if num_st != 1 else ic_center[0],
            ic_center[0] + t_mid,
            num_st,
        )
        for j in np.linspace(
            ic_center[1] - x_mid if num_sx != 1 else ic_center[1],
            ic_center[1] + x_mid,
            num_sx,
        )
        for k in np.linspace(
            ic_center[2] - y_mid if num_sy != 1 else ic_center[2],
            ic_center[2] + y_mid,
            num_sy,
        )
    ]
    
ics += [
    [
        sage.SR(specific_solution[i][0]),
        sage.SR(specific_solution[i][1]),
        sage.SR(specific_solution[i][2]),
    ]
    for i in range(len(specific_solution))
]

# initial condition points (to be graphed)
colors = cm.get_cmap("viridis")(np.linspace(0, 1, len(ics)))
ic_points = sum(
    [
        sage.point(
            [ics[i]],
            size=point_size * np.mean([(tmax - tmin), (xmax - xmin), (ymax - ymin)]),
            alpha=1,
            color=mcolors.to_hex(colors[i]),
            faceted=True,
            markeredgecolor="black",
        )
        for i in range(len(ics))
    ]
)

In [ ]:
def clip_trajectory_to_boundary(
    trajectory, tmin, tmax, xmin, xmax, ymin, ymax, compute_boundary=None
):
    clipped = []
    for point in trajectory:
        t_val, x_val, y_val = float(point[0]), float(point[1]), float(point[2])
        if compute_boundary is not None:
            if (
                abs(x_val) > compute_boundary
                or abs(y_val) > compute_boundary
                or abs(t_val) > compute_boundary
            ):
                break

        if (
            tmin - pad_t < t_val <= tmax + pad_t
            and xmin - pad_x < x_val <= xmax + pad_x
            and ymin - pad_y < y_val <= ymax + pad_y
        ):
            clipped.append(point)
    return clipped


def scaled_vector(t_val, x_val, y_val):
    vt, vx, vy = 1, f_x(t_val, x_val, y_val), g_y(t_val, x_val, y_val)
    mag_total = np.sqrt(vt**2 + vx**2 + vy**2)
    if mag_total == 0:
        return (0, 0, 0)
    return (vt * alpha / mag_total, vx * alpha / mag_total, vy * alpha / mag_total)

In [ ]:
t, x, y = var("t x y")

solutions, trajectory_vectors  = [], []
for idx, i in enumerate(ics):
    trajectory = sage.desolve_system_rk4(
        [f_x(t, x, y), g_y(t, x, y)],
        vars=[x, y],
        ivar=t,
        ics=i,
        end_points=[tmin, tmax + pad_t],
        step=step,
    )
    clipped = clip_trajectory_to_boundary(
        trajectory, tmin, tmax, xmin, xmax, ymin, ymax, compute_boundary
    )

    if len(clipped) >= 2:
        solutions.append(
            sage.list_plot(
                clipped,
                plotjoined=True,
                color=mcolors.to_hex(colors[idx]),
                thickness=1.5,
                # size=4 * np.mean([(tmax - tmin), (xmax - xmin), (ymax - ymin)]),
                # alpha=0.8,
                # faceted=True,
            )
        )
        trajectory_vectors.append((clipped, idx))

    print(f"Completed trajectory {idx + 1} of {len(ics)}")


In [ ]:
poincare_points_3d_up = []
poincare_points_3d_down = []
for traj, traj_idx in trajectory_vectors:
    for idx in range(len(traj) - 1):
        p1 = traj[idx]
        p2 = traj[idx + 1]
        t1, x1, y1 = float(p1[0]), float(p1[1]), float(p1[2])
        t2, x2, y2 = float(p2[0]), float(p2[1]), float(p2[2])
        h1 = poincare_a * t1 + poincare_b * x1 + poincare_c * y1 + poincare_d
        h2 = poincare_a * t2 + poincare_b * x2 + poincare_c * y2 + poincare_d
        if h1 * h2 < 0:
            alpha = -h1 / (h2 - h1)
            t_intersect = t1 + alpha * (t2 - t1)
            x_intersect = x1 + alpha * (x2 - x1)
            y_intersect = y1 + alpha * (y2 - y1)
            if h1 < -eps and h2 > eps:
                poincare_points_3d_up.append((t_intersect, x_intersect, y_intersect, traj_idx))
        elif abs(h1) <= eps:
            if idx > 0:
                if h2 > eps:
                    poincare_points_3d_up.append((t1, x1, y1, traj_idx))
                if h2 < -eps:
                    poincare_points_3d_down.append((t1, x1, y1, traj_idx))

print(
    f"Found {len(poincare_points_3d_up)} points on Poincaré section ('upward' crossings)"
)
print(
    f"Found {len(poincare_points_3d_down)} points on Poincaré section ('downward' crossings)"
)
print(
    f"\nPlane equation: {poincare_a}*t + {poincare_b}*x + {poincare_c}*y + {poincare_d} = 0"
)

In [ ]:
if len(poincare_points_3d_up) > 0:
    normal = np.array([poincare_a, poincare_b, poincare_c], dtype=float)
    normal = normal / np.linalg.norm(normal)
    if abs(normal[0]) < 0.9:
        v1 = np.array([1, 0, 0], dtype=float)
    else:
        v1 = np.array([0, 1, 0], dtype=float)

    u1 = v1 - np.dot(v1, normal) * normal
    u1 = u1 / np.linalg.norm(u1)
    u2 = np.cross(normal, u1)
    u2 = u2 / np.linalg.norm(u2)

    plane_2d_coords = []
    for t_val, x_val, y_val, idx in poincare_points_3d_up:
        point_3d = np.array([t_val, x_val, y_val])
        if abs(poincare_a) > 1e-10:
            ref_point = np.array([-poincare_d / poincare_a, 0, 0])
        elif abs(poincare_b) > 1e-10:
            ref_point = np.array([0, -poincare_d / poincare_b, 0])
        else:
            ref_point = np.array([0, 0, -poincare_d / poincare_c])
        vec_on_plane = point_3d - ref_point
        plane_2d_coords.append((np.dot(vec_on_plane, u1), np.dot(vec_on_plane, u2), idx))

    poincare_plot = plt.figure(figsize=(6, 6))
    u1_coords = [p[0]for p in (plane_2d_coords if not flip_x else reversed(copy.copy(plane_2d_coords)))]
    u2_coords = [p[1] for p in (plane_2d_coords if not flip_y else reversed(copy.copy(plane_2d_coords)))]
    color_idx = np.array([p[2] for p in plane_2d_coords])
    plt.scatter(
        u1_coords,
        u2_coords,
        s=80,
        alpha=1.0,
        c=colors[color_idx],
        edgecolors="black",
        linewidth=0.3,
    )
    plt.xlabel("u1", fontsize=8)
    plt.ylabel("u2", fontsize=8)
    plane_eq = (
        f"{poincare_a}*t + {poincare_b}*x + {poincare_c}*y + {poincare_d} = 0"
        if (poincare_a != 0 or poincare_b != 0 or poincare_c != 0)
        else "Poincaré Section"
    )
    plt.title(f"Poincaré Section (up): {plane_eq}", fontsize=10, fontweight="bold")
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig("./Plots/2D/poincare_plot_up.svg")
    plt.show()

    print(f"Plane normal vector: {normal}")
    print(f"Plane basis u1: {u1}")
    print(f"Plane basis u2: {u2}")
else:
    print("No points found on Poincaré section.")

In [ ]:
if len(poincare_points_3d_down) > 0:
    normal = np.array([poincare_a, poincare_b, poincare_c], dtype=float)
    normal = normal / np.linalg.norm(normal)
    if abs(normal[0]) < 0.9:
        v1 = np.array([1, 0, 0], dtype=float)
    else:
        v1 = np.array([0, 1, 0], dtype=float)

    u1 = v1 - np.dot(v1, normal) * normal
    u1 = u1 / np.linalg.norm(u1)
    u2 = np.cross(normal, u1) # flip sign to change orientation
    u2 = u2 / np.linalg.norm(u2)

    plane_2d_coords = []
    for t_val, x_val, y_val, idx in poincare_points_3d_down:
        point_3d = np.array([t_val, x_val, y_val])
        if abs(poincare_a) > 1e-10:
            ref_point = np.array([-poincare_d / poincare_a, 0, 0])
        elif abs(poincare_b) > 1e-10:
            ref_point = np.array([0, -poincare_d / poincare_b, 0])
        else:
            ref_point = np.array([0, 0, -poincare_d / poincare_c])
        vec_on_plane = point_3d - ref_point
        plane_2d_coords.append((np.dot(vec_on_plane, u1), np.dot(vec_on_plane, u2), idx))

    poincare_plot = plt.figure(figsize=(6, 6))
    u1_coords = [p[0] for p in plane_2d_coords]
    u2_coords = [p[1] for p in plane_2d_coords]
    color_idx = np.array([p[2] for p in plane_2d_coords])
    plt.scatter(
        u1_coords,
        u2_coords,
        s=80,
        alpha=1.0,
        c=colors[color_idx],
        edgecolors="black",
        linewidth=0.3,
    )
    plt.xlabel("u1", fontsize=8)
    plt.ylabel("u2", fontsize=8)
    plane_eq = (
        f"{poincare_a}*t + {poincare_b}*x + {poincare_c}*y + {poincare_d} = 0"
        if (poincare_a != 0 or poincare_b != 0 or poincare_c != 0)
        else "Poincaré Section"
    )
    plt.title(f"Poincaré Section (up): {plane_eq}", fontsize=10, fontweight="bold")
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig("./Plots/2D/poincare_plot_up.svg")
    plt.show()

    print(f"Plane normal vector: {normal}")
    print(f"Plane basis u1: {u1}")
    print(f"Plane basis u2: {u2}")
else:
    print("No points found on Poincaré section.")

In [ ]:
# Create the 3D plot

poincare_points_3d = poincare_points_3d_down + poincare_points_3d_up

t_axis = sage.line3d(
    [(tmin - pad_t, 0, 0), (tmax + pad_t, 0, 0)], color="black", thickness=1.5
)
x_axis = sage.line3d(
    [(0, xmin - pad_x, 0), (0, xmax + pad_x, 0)], color="black", thickness=1.5
)
y_axis = sage.line3d(
    [(0, 0, ymin - pad_y), (0, 0, ymax + pad_y)], color="black", thickness=1.5
)

t_label = sage.text3d("t", (tmax + pad_t + 1, 0, 0), color="red", fontsize=20)
x_label = sage.text3d("x", (0, xmax + pad_x + 1, 0), color="green", fontsize=20)
y_label = sage.text3d("y", (0, 0, ymax + pad_y + 1), color="blue", fontsize=20)

if show_vectors:
    vector_field = sage.plot_vector_field3d(
        (
            scaled_vector(t, x, y)[0],
            scaled_vector(t, x, y)[1],
            scaled_vector(t, x, y)[2],
        ),
        (t, tmin - pad_t, tmax + pad_t),
        (x, xmin - pad_x, xmax + pad_x),
        (y, ymin - pad_y, ymax + pad_y),
        plot_points=6,
        radius=np.mean([tmax - tmin, xmax - xmin, ymax - ymin]),
    )
    plot_obj = (
        sum(solutions)
        + ic_points
        + t_axis
        + x_axis
        + y_axis
        + t_label
        + x_label
        + y_label
        + vector_field
    )
else:
    plot_obj = (
        sum(solutions)
        + ic_points
        + t_axis
        + x_axis
        + y_axis
        + t_label
        + x_label
        + y_label
    )

if show_poincare_section:
    t_range = np.linspace(tmin, tmax + pad_t, 30)
    x_range = np.linspace(xmin - pad_x, xmax + pad_x, 30)
    y_range = np.linspace(ymin - pad_y, ymax + pad_y, 30)
    if poincare_a != 0:
        X, Y = np.meshgrid(x_range, y_range)
        T = -(poincare_b * X + poincare_c * Y + poincare_d) / poincare_a
        T = np.clip(T, tmin, tmax + pad_t)
    elif poincare_b != 0:
        T, Y = np.meshgrid(t_range, y_range)
        X = -(poincare_a * T + poincare_c * Y + poincare_d) / poincare_b
        X = np.clip(X, xmin - pad_x, xmax + pad_x)
    elif poincare_c != 0:
        T, X = np.meshgrid(t_range, x_range)
        Y = -(poincare_a * T + poincare_b * X + poincare_d) / poincare_c
        Y = np.clip(Y, ymin - pad_y, ymax + pad_y)

    plane_lines = []
    for i in range(T.shape[0]):
        points = [[T[i, j], X[i, j], Y[i, j]] for j in range(T.shape[1])]
        if len(points) > 1:
            plane_lines.append(
                sage.line3d(points, color="red", thickness=1.5, alpha=0.3)
            )
    for j in range(T.shape[1]):
        points = [[T[i, j], X[i, j], Y[i, j]] for i in range(T.shape[0])]
        if len(points) > 1:
            plane_lines.append(
                sage.line3d(points, color="red", thickness=1.5, alpha=0.3)
            )
    for line in plane_lines:
        plot_obj += line
    for point in poincare_points_3d:
        plot_obj += sage.point(
            [(point[0], point[1], point[2])],
            size=point_size * np.mean([(tmax - tmin), (xmax - xmin), (ymax - ymin)]),
            color=mcolors.to_hex(colors[point[3]]),
        )


plot_obj.save("./Plots/3D/3D_2SYS_ODE.html", viewer="threejs", online=True)
print("Plot saved to 3D_2SYS_ODE.html")